# Détection de bots Twitter

Ce notebook construit une pipeline de détection de bots **au niveau utilisateur** à partir des six datasets d'entraînement.


## 1. Imports et configuration


In [1]:
from pathlib import Path
import json
import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS_STACK = 5

# Scoring officiel de la compétition
TP_REWARD = 2
FN_PENALTY = -2
FP_PENALTY = -6

DATA_DIR = Path("data/train")

JSON_FILES = [
    DATA_DIR / "dataset.posts&users.1.json",
    DATA_DIR / "dataset.posts&users.2.json",
    DATA_DIR / "dataset.posts&users.3.json",
    DATA_DIR / "dataset.posts&users.4.json",
    DATA_DIR / "dataset.posts&users.5.json",
    DATA_DIR / "dataset.posts&users.6.json",
]

BOT_FILES = {
    1: DATA_DIR / "dataset.bots.1.txt",
    2: DATA_DIR / "dataset.bots.2.txt",
    3: DATA_DIR / "dataset.bots.3.txt",
    4: DATA_DIR / "dataset.bots.4.txt",
    5: DATA_DIR / "dataset.bots.5.txt",
    6: DATA_DIR / "dataset.bots.6.txt",
}

TARGET_COL = "is_bot"
ID_COL = "user_id"

# Colonnes conservées pour l'analyse / validation, mais exclues de l'apprentissage
LEAKAGE_COLS = ["dataset_id", "dataset_lang"]

pd.set_option("display.max_columns", 200)

print("Device embeddings :", DEVICE)
print("DATA_DIR          :", DATA_DIR.resolve())
print("Nb datasets train :", len(JSON_FILES))

Device embeddings : cuda
DATA_DIR          : C:\Users\amzid\bot_competitions\Botornobot\data\train
Nb datasets train : 6


## 2. Chargement et préparation des données


In [2]:
def competition_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()

    return TP_REWARD * tp + FN_PENALTY * fn + FP_PENALTY * fp


def load_competition_json(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    metadata = {
        "dataset_id": data.get("id"),
        "lang": data.get("lang"),
        **data.get("metadata", {}),
    }

    posts_df = pd.DataFrame(data.get("posts", []))
    users_df = pd.DataFrame(data.get("users", []))

    expected_posts = {"text", "created_at", "id", "author_id", "lang"}
    expected_users = {"id", "username", "name", "description", "location", "tweet_count", "z_score"}

    missing_posts = expected_posts - set(posts_df.columns)
    missing_users = expected_users - set(users_df.columns)

    if missing_posts:
        raise ValueError(f"Colonnes manquantes dans posts: {missing_posts}")
    if missing_users:
        raise ValueError(f"Colonnes manquantes dans users: {missing_users}")

    posts_df["created_at"] = pd.to_datetime(posts_df["created_at"], utc=True, errors="coerce")

    return metadata, posts_df, users_df


def load_bot_ids(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

In [3]:
all_posts = []
all_users = []
dataset_summaries = []

for json_path in JSON_FILES:
    metadata, posts_df, users_df = load_competition_json(json_path)
    bot_ids = load_bot_ids(BOT_FILES[metadata["dataset_id"]])

    users_df = users_df.copy()
    posts_df = posts_df.copy()

    users_df["id"] = users_df["id"].astype(str)
    posts_df["author_id"] = posts_df["author_id"].astype(str)

    users_df["is_bot"] = users_df["id"].isin(bot_ids).astype(int)
    users_df["dataset_id"] = metadata["dataset_id"]
    users_df["dataset_lang"] = metadata["lang"]

    posts_df["dataset_id"] = metadata["dataset_id"]
    posts_df["dataset_lang"] = metadata["lang"]

    all_posts.append(posts_df)
    all_users.append(users_df)

    dataset_summaries.append({
        "dataset_id": metadata["dataset_id"],
        "lang": metadata["lang"],
        "users": len(users_df),
        "posts": len(posts_df),
        "bots": int(users_df["is_bot"].sum()),
        "bot_ratio": users_df["is_bot"].mean(),
    })

posts = pd.concat(all_posts, ignore_index=True)
users = pd.concat(all_users, ignore_index=True)

summary_df = pd.DataFrame(dataset_summaries).sort_values("dataset_id").reset_index(drop=True)
display(summary_df)

print("posts shape:", posts.shape)
print("users shape:", users.shape)

,dataset_id,lang,users,posts,bots,bot_ratio
0,1,en,262,6645,54,0.206107
1,2,fr,171,4643,27,0.157895
2,3,en,257,7255,51,0.198444
3,4,fr,172,4361,28,0.162791
4,5,en,426,14080,53,0.124413
5,6,fr,283,8192,28,0.098940


posts shape: (45176, 7)
users shape: (1571, 10)


## 3. Vérifications de cohérence


In [4]:

print("Nombre total d'utilisateurs :", len(users))
print("Nombre total de bots        :", int(users[TARGET_COL].sum()))
print("Ratio de bots               :", users[TARGET_COL].mean())
print()

# Répartition par langue
lang_stats = (
    users.groupby("dataset_lang")[TARGET_COL]
    .agg(["count", "sum"])
    .rename(columns={"count": "users", "sum": "bots"})
)
lang_stats["bot_ratio"] = lang_stats["bots"] / lang_stats["users"]

display(lang_stats)

# Répartition par dataset
dataset_stats = (
    users.groupby("dataset_id")[TARGET_COL]
    .agg(["count", "sum"])
    .rename(columns={"count": "users", "sum": "bots"})
)
dataset_stats["bot_ratio"] = dataset_stats["bots"] / dataset_stats["users"]

display(dataset_stats)

# Vérification cohérence posts/users
missing_authors = set(posts["author_id"].astype(str)) - set(users["id"].astype(str))
print("\nNombre d'author_id absents de users :", len(missing_authors))

if len(missing_authors) > 0:
    print("Attention: certains posts n'ont pas de user correspondant")


Nombre total d'utilisateurs : 1571
Nombre total de bots        : 241
Ratio de bots               : 0.1534054742202419



,users,bots,bot_ratio
dataset_lang,,,
en,945,158,0.167196
fr,626,83,0.132588


,users,bots,bot_ratio
dataset_id,,,
1,262,54,0.206107
2,171,27,0.157895
3,257,51,0.198444
4,172,28,0.162791
5,426,53,0.124413
6,283,28,0.098940



Nombre d'author_id absents de users : 0


### Vérifications complémentaires

On ajoute ici quelques contrôles rapides pour éviter les surprises : doublons d'identifiants utilisateur, valeurs manquantes dans les colonnes critiques et cohérence minimale entre tables.


In [5]:
print("Doublons user_id dans users :", users["id"].astype(str).duplicated().sum())
print("Doublons post id dans posts :", posts["id"].astype(str).duplicated().sum())

critical_user_cols = ["id", "username", "tweet_count", "z_score"]
critical_post_cols = ["id", "author_id", "text", "created_at"]

print("\nNA users:")
display(users[critical_user_cols].isna().sum().rename("na_count").to_frame())

print("NA posts:")
display(posts[critical_post_cols].isna().sum().rename("na_count").to_frame())

# Vérification supplémentaire utile
empty_text_count = posts["text"].fillna("").astype(str).str.strip().eq("").sum()
print("Posts avec texte vide :", int(empty_text_count))

missing_created_at = posts["created_at"].isna().sum()
print("Posts avec created_at invalide :", int(missing_created_at))

Doublons user_id dans users : 0
Doublons post id dans posts : 0

NA users:


,na_count
id,0
username,0
tweet_count,0
z_score,0


NA posts:


,na_count
id,0
author_id,0
text,0
created_at,0


Posts avec texte vide : 0
Posts avec created_at invalide : 0


## 4. Construction des variables agrégées au niveau utilisateur


In [6]:
HASHTAG_RE = re.compile(r"#\w+")
MENTION_RE = re.compile(r"@mention|\B@\w+")
URL_RE = re.compile(r"https?://\S+|twitter_link")
DIGIT_RE = re.compile(r"\d")


def lexical_diversity(texts):
    tokens = []
    for t in texts:
        if pd.isna(t):
            continue
        tokens.extend(str(t).lower().split())
    return len(set(tokens)) / len(tokens) if tokens else 0.0


def duplicate_ratio(texts):
    cleaned = [
        str(t).strip().lower()
        for t in texts
        if pd.notna(t) and str(t).strip()
    ]
    return 1.0 - len(set(cleaned)) / len(cleaned) if cleaned else 0.0


def entropy_from_counts(counts):
    total = sum(counts)
    if total == 0:
        return 0.0
    probs = [c / total for c in counts if c > 0]
    return -sum(p * math.log(p, 2) for p in probs)


def burstiness(values):
    if len(values) == 0:
        return 0.0
    mu = float(np.mean(values))
    sigma = float(np.std(values))
    return 0.0 if (sigma + mu) == 0 else (sigma - mu) / (sigma + mu)


def char_entropy(texts):
    full = " ".join([str(t) for t in texts if pd.notna(t)])
    if not full:
        return 0.0
    counts = Counter(full)
    total = sum(counts.values())
    probs = [c / total for c in counts.values()]
    return -sum(p * math.log(p, 2) for p in probs)


def first_token_repetition_ratio(texts):
    toks = []
    for t in texts:
        if pd.isna(t):
            continue
        s = str(t).strip().lower().split()
        if s:
            toks.append(s[0])
    return max(Counter(toks).values()) / len(toks) if toks else 0.0


def exact_duplicate_ratio(texts):
    cleaned = [
        str(t).strip().lower()
        for t in texts
        if pd.notna(t) and str(t).strip()
    ]
    return 1.0 - len(set(cleaned)) / len(cleaned) if cleaned else 0.0

In [7]:
def build_user_level_features(posts_df, users_df):
    u = users_df.copy()
    p = posts_df.copy()

    u["id"] = u["id"].astype(str)
    u["username"] = u["username"].fillna("")
    u["name"] = u["name"].fillna("")
    u["description"] = u["description"].fillna("")
    u["location"] = u["location"].fillna("")

    profile = pd.DataFrame({
        "user_id": u["id"],
        "dataset_id": u["dataset_id"],
        "dataset_lang": u["dataset_lang"],
        "is_bot": u["is_bot"].astype(int),
        "username_len": u["username"].str.len(),
        "username_has_digit": u["username"].apply(lambda x: int(bool(DIGIT_RE.search(str(x))))),
        "name_len": u["name"].str.len(),
        "description_len": u["description"].str.len(),
        "description_is_empty": (u["description"].str.strip() == "").astype(int),
        "location_len": u["location"].str.len(),
        "location_is_empty": (u["location"].str.strip() == "").astype(int),
        "tweet_count_declared": pd.to_numeric(u["tweet_count"], errors="coerce").fillna(0),
        "tweet_count_log": np.log1p(pd.to_numeric(u["tweet_count"], errors="coerce").fillna(0)),
        "z_score": pd.to_numeric(u["z_score"], errors="coerce").fillna(0.0),
    })

    p["text"] = p["text"].fillna("")
    p["author_id"] = p["author_id"].astype(str)
    p["text_len"] = p["text"].str.len()
    p["num_tokens"] = p["text"].apply(lambda x: len(str(x).split()))
    p["num_hashtags"] = p["text"].apply(lambda x: len(HASHTAG_RE.findall(str(x))))
    p["num_mentions"] = p["text"].apply(lambda x: len(MENTION_RE.findall(str(x))))
    p["num_urls"] = p["text"].apply(lambda x: len(URL_RE.findall(str(x))))
    p["hour"] = p["created_at"].dt.hour.fillna(-1).astype(int)

    rows = []

    for user_id, grp in p.groupby("author_id", sort=False):
        grp = grp.sort_values("created_at")
        texts = grp["text"].tolist()
        hours = grp["hour"].tolist()

        clean_texts = [
            str(t).strip().lower()
            for t in texts
            if pd.notna(t) and str(t).strip()
        ]

        timestamps = grp["created_at"].dropna().astype("int64") // 10**9
        deltas = np.diff(timestamps.to_numpy()) if len(timestamps) >= 2 else np.array([])

        hour_counts = Counter(h for h in hours if h >= 0)
        hour_dist = [hour_counts.get(h, 0) for h in range(24)]

        rows.append({
            "user_id": user_id,
            "tweet_count_observed": len(grp),
            "avg_text_len": grp["text_len"].mean(),
            "std_text_len": grp["text_len"].std(ddof=0),
            "avg_num_tokens": grp["num_tokens"].mean(),
            "avg_num_hashtags": grp["num_hashtags"].mean(),
            "avg_num_mentions": grp["num_mentions"].mean(),
            "avg_num_urls": grp["num_urls"].mean(),
            "lexical_diversity": lexical_diversity(texts),
            "duplicate_ratio": duplicate_ratio(texts),
            "exact_duplicate_ratio": exact_duplicate_ratio(texts),
            "hour_entropy": entropy_from_counts(hour_dist),
            "night_tweet_ratio": np.mean([h in {0, 1, 2, 3, 4, 5} for h in hours]) if hours else 0.0,
            "avg_delta_seconds": float(np.mean(deltas)) if len(deltas) else 0.0,
            "std_delta_seconds": float(np.std(deltas)) if len(deltas) else 0.0,
            "burstiness_delta": burstiness(deltas) if len(deltas) else 0.0,
            "text_unique_ratio": len(set(clean_texts)) / len(clean_texts) if clean_texts else 0.0,
            "short_tweet_ratio": np.mean([len(str(t).split()) <= 4 for t in texts]) if texts else 0.0,
            "url_tweet_ratio": np.mean([len(URL_RE.findall(str(t))) > 0 for t in texts]) if texts else 0.0,
            "hashtag_tweet_ratio": np.mean([len(HASHTAG_RE.findall(str(t))) > 0 for t in texts]) if texts else 0.0,
            "mention_tweet_ratio": np.mean([len(MENTION_RE.findall(str(t))) > 0 for t in texts]) if texts else 0.0,
            "char_entropy": char_entropy(texts),
            "first_token_repetition_ratio": first_token_repetition_ratio(texts),
            "avg_word_len": np.mean([
                np.mean([len(w) for w in str(t).split()]) if str(t).split() else 0.0
                for t in texts
            ]) if texts else 0.0,
        })

    temporal_text = pd.DataFrame(rows)
    final_df = profile.merge(temporal_text, on="user_id", how="left")

    for col in final_df.columns:
        if col not in {"user_id", "dataset_lang"} and final_df[col].dtype != "O":
            final_df[col] = final_df[col].fillna(0)

    return final_df


df = build_user_level_features(posts, users)
print("Shape user-level:", df.shape)
display(df.head())

FEATURE_COLS = [c for c in df.columns if c not in [TARGET_COL, ID_COL] + LEAKAGE_COLS]
X_tab_full = df[FEATURE_COLS].copy()
y = df[TARGET_COL].astype(int).values

categorical_cols = [c for c in FEATURE_COLS if X_tab_full[c].dtype == "object"]
numeric_cols = [c for c in FEATURE_COLS if c not in categorical_cols]

print("Nombre de variables utilisées :", len(FEATURE_COLS))
print("Variables exclues de l'apprentissage :", LEAKAGE_COLS)

Shape user-level: (1571, 37)


,user_id,dataset_id,dataset_lang,is_bot,username_len,username_has_digit,name_len,description_len,description_is_empty,location_len,location_is_empty,tweet_count_declared,tweet_count_log,z_score,tweet_count_observed,avg_text_len,std_text_len,avg_num_tokens,avg_num_hashtags,avg_num_mentions,avg_num_urls,lexical_diversity,duplicate_ratio,exact_duplicate_ratio,hour_entropy,night_tweet_ratio,avg_delta_seconds,std_delta_seconds,burstiness_delta,text_unique_ratio,short_tweet_ratio,url_tweet_ratio,hashtag_tweet_ratio,mention_tweet_ratio,char_entropy,first_token_repetition_ratio,avg_word_len
0,dd6de229-7892-4bec-b13d-6769c3dbd8ec,1,en,0,11,0,11,116,0,27,0,39,3.688879,0.708294,39,141.230769,82.163209,22.717949,0.000000,0.000000,0.948718,0.528217,0.0,0.0,1.868525,0.102564,4545.289474,16530.085006,0.568663,1.0,0.000000,0.948718,0.000000,0.000000,4.662888,0.153846,5.829250
1,475042a3-4676-46ad-af17-0341f1dfc58a,1,en,0,8,1,10,138,0,12,0,29,3.401197,0.188918,29,60.551724,32.641824,11.344828,0.000000,0.000000,0.068966,0.574468,0.0,0.0,2.373560,0.241379,5748.535714,13466.581326,0.401665,1.0,0.103448,0.068966,0.000000,0.000000,4.626254,0.172414,4.571830
2,d03ae074-a881-4c85-9ea0-378bf39ede7a,1,en,0,7,1,16,146,0,10,0,59,4.094345,1.747045,59,151.254237,54.063238,21.711864,1.593220,0.000000,0.694915,0.380952,0.0,0.0,3.547921,0.169492,2947.965517,8283.760957,0.475065,1.0,0.033898,0.694915,0.966102,0.000000,5.033623,0.864407,6.030782
3,eed74609-b31d-4337-ba00-a338f9f65dbe,1,en,0,11,0,43,158,0,12,0,37,3.637586,0.604419,37,48.756757,10.455583,8.054054,0.189189,0.000000,0.000000,0.436242,0.0,0.0,2.619391,0.081081,4768.472222,12539.596193,0.448989,1.0,0.000000,0.000000,0.189189,0.000000,5.103231,0.432432,4.840862
4,fbc68db2-e072-49c4-af92-94454838492d,1,en,0,11,0,12,49,0,13,0,14,2.708050,-0.590146,14,93.000000,62.517426,15.785714,0.000000,0.142857,0.357143,0.719457,0.0,0.0,2.692381,0.428571,13171.384615,20389.422101,0.215073,1.0,0.000000,0.285714,0.000000,0.142857,4.865224,0.142857,5.078089


Nombre de variables utilisées : 33
Variables exclues de l'apprentissage : ['dataset_id', 'dataset_lang']


### Variables retenues pour l'apprentissage

Les colonnes `dataset_id` et `dataset_lang` sont volontairement exclues de l'apprentissage tabulaire pour éviter d'apprendre artificiellement l'identité des datasets. Elles restent disponibles pour l'analyse et la validation leave-one-dataset-out.


In [8]:
FEATURE_COLS = [c for c in df.columns if c not in [TARGET_COL, ID_COL] + LEAKAGE_COLS]

X_tab_full = df[FEATURE_COLS].copy()

# catégorielles = dtype object OU category
categorical_cols = [
    c for c in FEATURE_COLS
    if X_tab_full[c].dtype == "object" or str(X_tab_full[c].dtype) == "category"
]

numeric_cols = [c for c in FEATURE_COLS if c not in categorical_cols]

print("Nombre de variables utilisées :", len(FEATURE_COLS))
print("Variables catégorielles :", len(categorical_cols))
print("Variables numériques    :", len(numeric_cols))

display(pd.DataFrame({
    "feature": FEATURE_COLS,
    "dtype": X_tab_full.dtypes.astype(str)
}).head(30))

Nombre de variables utilisées : 33
Variables catégorielles : 0
Variables numériques    : 33


,feature,dtype
username_len,username_len,int64
username_has_digit,username_has_digit,int64
name_len,name_len,int64
description_len,description_len,int64
description_is_empty,description_is_empty,int64
location_len,location_len,int64
location_is_empty,location_is_empty,int64
tweet_count_declared,tweet_count_declared,int64
tweet_count_log,tweet_count_log,float64
z_score,z_score,float64


## 5. Jeux textuels et embeddings


In [9]:
def build_user_text_dataset(posts_df, user_order_df):
    p = posts_df.copy()
    p["text"] = p["text"].fillna("").astype(str)
    p["author_id"] = p["author_id"].astype(str)

    user_text = (
        p.groupby("author_id")["text"]
        .apply(lambda x: " ".join(x.tolist()))
        .reset_index()
        .rename(columns={"author_id": "user_id", "text": "all_text"})
    )

    out = user_order_df.copy()

    # robustesse : accepte soit "id", soit "user_id"
    if "user_id" not in out.columns and "id" in out.columns:
        out = out.rename(columns={"id": "user_id"})

    keep_cols = ["user_id"]
    if TARGET_COL in out.columns:
        keep_cols.append(TARGET_COL)

    out = out[keep_cols].copy()
    out = out.merge(user_text, on="user_id", how="left")
    out["all_text"] = out["all_text"].fillna("")

    return out


def build_user_embedding_dataset(posts_df, users_df, model, batch_size=64):
    p = posts_df.copy()
    p["text"] = p["text"].fillna("").astype(str)
    p["author_id"] = p["author_id"].astype(str)

    emb_dim = model.get_sentence_embedding_dimension()
    rows = []

    for user_id, grp in p.groupby("author_id", sort=False):
        texts = [t for t in grp["text"].tolist() if t.strip()]

        if len(texts) == 0:
            user_vec = np.zeros(emb_dim, dtype=np.float32)
        else:
            embs = model.encode(
                texts,
                batch_size=batch_size,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False,
            )
            user_vec = embs.mean(axis=0).astype(np.float32)

        row = {"user_id": user_id}
        for i, val in enumerate(user_vec):
            row[f"emb_{i}"] = val
        rows.append(row)

    emb_df = pd.DataFrame(rows)

    base_users = users_df.copy()
    if "user_id" not in base_users.columns and "id" in base_users.columns:
        base_users = base_users.rename(columns={"id": "user_id"})

    keep_cols = ["user_id"]
    if TARGET_COL in base_users.columns:
        keep_cols.append(TARGET_COL)

    base_users = base_users[keep_cols].copy()

    emb_df = base_users.merge(emb_df, on="user_id", how="left")

    emb_cols = [c for c in emb_df.columns if c.startswith("emb_")]
    emb_df[emb_cols] = emb_df[emb_cols].fillna(0.0)

    return emb_df

In [10]:
text_df = build_user_text_dataset(posts, df)

embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=DEVICE,
)
print("Embedding dimension:", embed_model.get_sentence_embedding_dimension())

embed_df = build_user_embedding_dataset(posts, df, embed_model, batch_size=64)

X_text_all = text_df["all_text"].copy()
X_emb_all = embed_df.drop(columns=["user_id", TARGET_COL]).copy()

display(text_df.head())
print("Shape embed_df:", embed_df.shape)
display(embed_df.head())

Embedding dimension: 384


,user_id,is_bot,all_text
0,dd6de229-7892-4bec-b13d-6769c3dbd8ec,0,What is this show even about? https://t.co/twi...
1,475042a3-4676-46ad-af17-0341f1dfc58a,0,You should check my bio and follow The governm...
2,d03ae074-a881-4c85-9ea0-378bf39ede7a,0,LIVE #NWSL on Prime Video 2024 #NWSLKickoff 20...
3,eed74609-b31d-4337-ba00-a338f9f65dbe,0,NBA Play:\n\nJazz +2\n\nMoving CBB Play:\n\n62...
4,fbc68db2-e072-49c4-af92-94454838492d,0,WWE Smackdown is on tv! Let’s go! The Rock has...


Shape embed_df: (1571, 386)


,user_id,is_bot,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,emb_32,emb_33,emb_34,emb_35,emb_36,emb_37,emb_38,emb_39,emb_40,emb_41,emb_42,emb_43,emb_44,emb_45,emb_46,emb_47,emb_48,emb_49,emb_50,emb_51,emb_52,emb_53,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63,emb_64,emb_65,emb_66,emb_67,emb_68,emb_69,emb_70,emb_71,emb_72,emb_73,emb_74,emb_75,emb_76,emb_77,emb_78,emb_79,emb_80,emb_81,emb_82,emb_83,emb_84,emb_85,emb_86,emb_87,emb_88,emb_89,emb_90,emb_91,emb_92,emb_93,emb_94,emb_95,emb_96,emb_97,...,emb_284,emb_285,emb_286,emb_287,emb_288,emb_289,emb_290,emb_291,emb_292,emb_293,emb_294,emb_295,emb_296,emb_297,emb_298,emb_299,emb_300,emb_301,emb_302,emb_303,emb_304,emb_305,emb_306,emb_307,emb_308,emb_309,emb_310,emb_311,emb_312,emb_313,emb_314,emb_315,emb_316,emb_317,emb_318,emb_319,emb_320,emb_321,emb_322,emb_323,emb_324,emb_325,emb_326,emb_327,emb_328,emb_329,emb_330,emb_331,emb_332,emb_333,emb_334,emb_335,emb_336,emb_337,emb_338,emb_339,emb_340,emb_341,emb_342,emb_343,emb_344,emb_345,emb_346,emb_347,emb_348,emb_349,emb_350,emb_351,emb_352,emb_353,emb_354,emb_355,emb_356,emb_357,emb_358,emb_359,emb_360,emb_361,emb_362,emb_363,emb_364,emb_365,emb_366,emb_367,emb_368,emb_369,emb_370,emb_371,emb_372,emb_373,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,dd6de229-7892-4bec-b13d-6769c3dbd8ec,0,-0.030293,0.034759,0.006328,-0.104312,0.003025,-0.070719,0.134285,-0.018340,0.116184,-0.076170,0.161020,-0.029840,0.080771,0.016760,-0.041290,-0.109968,0.027983,-0.022206,-0.190095,0.098255,-0.095093,-0.063704,0.125541,0.061298,0.015197,-0.127050,-0.090923,-0.004498,-0.259331,0.030085,0.019056,-0.068784,-0.135741,0.068720,-0.080202,0.048153,0.048937,0.066982,0.024046,-0.049492,-0.054287,-0.064198,-0.021105,0.141862,0.038574,0.037713,-0.059412,-0.021142,-0.126431,-0.024945,-0.079094,0.057153,-0.179259,-0.092611,0.043537,-0.046006,-0.009752,0.064643,0.024718,0.021524,0.002716,-0.104587,-0.005577,0.124078,-0.042104,-0.074366,-0.050343,-0.060587,-0.104650,0.094322,0.092799,0.036335,0.016048,0.147187,0.144462,-0.022685,0.000673,-0.060888,-0.037951,-0.119141,0.047590,-0.037054,-0.038006,-0.027691,-0.024494,-0.024634,-0.085321,-0.020577,0.001308,0.019227,-0.138153,-0.111986,0.234942,0.039447,0.103551,0.040553,-0.082206,0.011020,...,0.026127,0.030263,-0.007150,-0.022579,0.062767,0.039593,0.021575,0.177324,-0.007051,0.096403,0.021148,-0.025789,-0.076308,0.033750,0.040354,0.164653,0.006680,0.111558,-0.010814,-0.043707,0.212823,-0.059441,-0.064276,-0.011165,0.110361,0.117604,-0.009861,0.076863,-0.101204,0.103340,0.102211,0.108233,-0.023049,0.194831,0.092067,-0.192349,0.014683,-0.063540,-0.143539,0.105020,0.109535,0.081589,0.018578,-0.260537,-0.031203,0.142981,-0.075055,0.063187,-0.056829,-0.091822,0.170738,0.103961,-0.235931,-0.231748,0.049305,0.120580,-0.008376,0.004014,0.016372,0.019787,-0.132860,0.035464,-0.122459,0.124508,0.120023,0.042983,0.096912,0.250858,-0.291997,0.072046,-0.127768,-0.047936,0.022859,-0.117610,0.263150,-0.056476,0.053421,-0.031835,0.024133,0.357512,-0.059893,0.061056,0.034863,0.028317,-0.230689,-0.036936,-0.019047,0.091961,-0.027943,-0.044592,0.132679,-0.093582,0.013122,0.115306,-0.184711,0.027198,0.213085,-0.016762,0.023907,0.025354
1,475042a3-4676-46ad-af17-0341f1dfc58a,0,0.063320,0.032441,0.035082,0.093238,0.025785,-0.067906,0.151203,0.041844,-0.083995,0.006208,0.087852,0.002471,-0.003072,0.002659,-0.015911,0.004596,0.020247,-0.174379,-0.180240,0.118786,-0.307122,-0.103251,-0.056106,0.045573,0.145857,0.073474,-0.015083,0.076171,-0.204889,0.003474,-0.013668,-0.142786,0.037320,0.078403,0.068096,0.152028,-0.006930,0.072166,0.070631,-0.022293,-0.029422,-0.135380,-0.104729,0.091221,-0.005887,-0.105395,0.057230,-0.054442,0.174507,-0.131001,-0.085693,0.007949,-0.109109,-0.081189,0.118322,-0.038659,0.103649,0.0

## 6. Fabriques de modèles et fonctions d'évaluation


In [11]:
def make_tabular_baseline(categorical_cols, numeric_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_cols,
            ),
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced")),
    ])


def make_lgbm_model(random_state=RANDOM_STATE):
    return LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=10,
        subsample=0.9,
        colsample_bytree=0.9,
        class_weight="balanced",
        random_state=random_state,
        verbose=-1,
    )


def make_text_model():
    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=30000,
            )
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
            )
        )
    ])


def make_embed_clf():
    return LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
    )


def find_best_threshold(y_true, y_prob, start=0.01, stop=0.99, step=0.01):
    thresholds = np.arange(start, stop + 1e-9, step)

    best_t = 0.5
    best_score = -10**18
    rows = []

    for t in thresholds:
        pred_t = (y_prob >= t).astype(int)

        score = competition_score(y_true, pred_t)
        f1 = f1_score(y_true, pred_t, zero_division=0)
        p = precision_score(y_true, pred_t, zero_division=0)
        r = recall_score(y_true, pred_t, zero_division=0)

        rows.append((t, score, f1, p, r))

        if score > best_score:
            best_score = score
            best_t = t

    threshold_df = pd.DataFrame(
        rows,
        columns=["threshold", "score", "f1", "precision", "recall"]
    )
    return best_t, best_score, threshold_df


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "score": competition_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }, y_pred


def fit_predict_tabular(X_train_tab, y_train, X_valid_tab, categorical_cols):
    X_train_tab = X_train_tab.copy()
    X_valid_tab = X_valid_tab.copy()

    for col in categorical_cols:
        X_train_tab[col] = X_train_tab[col].astype("category")
        X_valid_tab[col] = pd.Categorical(
            X_valid_tab[col],
            categories=X_train_tab[col].cat.categories
        )

    model = make_lgbm_model()
    model.fit(X_train_tab, y_train, categorical_feature=categorical_cols)
    valid_prob = model.predict_proba(X_valid_tab)[:, 1]
    return model, valid_prob


def generate_oof_meta_features(
    X_tab,
    X_text,
    X_emb,
    y,
    categorical_cols,
    n_splits=N_SPLITS_STACK,
    random_state=RANDOM_STATE,
):
    X_tab = X_tab.reset_index(drop=True)
    X_text = X_text.reset_index(drop=True)
    X_emb = X_emb.reset_index(drop=True)
    y = np.asarray(y)

    oof_tab = np.zeros(len(y))
    oof_text = np.zeros(len(y))
    oof_emb = np.zeros(len(y))

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.arange(len(y)), y), start=1):
        print(f"  Fold interne {fold}/{n_splits}")

        y_tr = y[tr_idx]

        _, tab_prob = fit_predict_tabular(
            X_tab.iloc[tr_idx],
            y_tr,
            X_tab.iloc[va_idx],
            categorical_cols,
        )
        oof_tab[va_idx] = tab_prob

        text_model = make_text_model()
        text_model.fit(X_text.iloc[tr_idx], y_tr)
        oof_text[va_idx] = text_model.predict_proba(X_text.iloc[va_idx])[:, 1]

        emb_model = make_embed_clf()
        emb_model.fit(X_emb.iloc[tr_idx], y_tr)
        oof_emb[va_idx] = emb_model.predict_proba(X_emb.iloc[va_idx])[:, 1]

    return pd.DataFrame({
        "tabular_prob": oof_tab,
        "text_prob": oof_text,
        "embed_prob": oof_emb,
    })


def fit_base_models_and_predict_valid(
    X_train_tab,
    X_valid_tab,
    X_train_text,
    X_valid_text,
    X_train_emb,
    X_valid_emb,
    y_train,
    categorical_cols,
):
    _, tab_prob_valid = fit_predict_tabular(X_train_tab, y_train, X_valid_tab, categorical_cols)

    text_model = make_text_model()
    text_model.fit(X_train_text, y_train)
    text_prob_valid = text_model.predict_proba(X_valid_text)[:, 1]

    emb_model = make_embed_clf()
    emb_model.fit(X_train_emb, y_train)
    emb_prob_valid = emb_model.predict_proba(X_valid_emb)[:, 1]

    return pd.DataFrame({
        "tabular_prob": tab_prob_valid,
        "text_prob": text_prob_valid,
        "embed_prob": emb_prob_valid,
    })

## 7. Split aléatoire exploratoire


In [12]:
indices = np.arange(len(df))
train_idx, valid_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_tab = X_tab_full.iloc[train_idx].reset_index(drop=True)
X_valid_tab = X_tab_full.iloc[valid_idx].reset_index(drop=True)
X_train_text = X_text_all.iloc[train_idx].reset_index(drop=True)
X_valid_text = X_text_all.iloc[valid_idx].reset_index(drop=True)
X_train_emb = X_emb_all.iloc[train_idx].reset_index(drop=True)
X_valid_emb = X_emb_all.iloc[valid_idx].reset_index(drop=True)

y_train = y[train_idx]
y_valid = y[valid_idx]

# Baseline logistique tabulaire
baseline_model = make_tabular_baseline(categorical_cols, numeric_cols)
baseline_model.fit(X_train_tab, y_train)
baseline_prob = baseline_model.predict_proba(X_valid_tab)[:, 1]
baseline_best_t, baseline_best_score, _ = find_best_threshold(y_valid, baseline_prob)
baseline_metrics, baseline_pred = compute_metrics(y_valid, baseline_prob, threshold=baseline_best_t)

# LightGBM tabulaire
lgbm_model, lgbm_prob = fit_predict_tabular(X_train_tab, y_train, X_valid_tab, categorical_cols)
lgbm_best_t, lgbm_best_score, _ = find_best_threshold(y_valid, lgbm_prob)
lgbm_metrics, lgbm_pred = compute_metrics(y_valid, lgbm_prob, threshold=lgbm_best_t)

# TF-IDF
tfidf_model = make_text_model()
tfidf_model.fit(X_train_text, y_train)
tfidf_prob = tfidf_model.predict_proba(X_valid_text)[:, 1]
tfidf_best_t, tfidf_best_score, _ = find_best_threshold(y_valid, tfidf_prob)
tfidf_metrics, tfidf_pred = compute_metrics(y_valid, tfidf_prob, threshold=tfidf_best_t)

# Embeddings
emb_model = make_embed_clf()
emb_model.fit(X_train_emb, y_train)
emb_prob = emb_model.predict_proba(X_valid_emb)[:, 1]
emb_best_t, emb_best_score, _ = find_best_threshold(y_valid, emb_prob)
emb_metrics, emb_pred = compute_metrics(y_valid, emb_prob, threshold=emb_best_t)

results_single_split = pd.DataFrame([
    {"model": "Baseline LogReg", "threshold": baseline_best_t, **baseline_metrics},
    {"model": "LightGBM", "threshold": lgbm_best_t, **lgbm_metrics},
    {"model": "TF-IDF", "threshold": tfidf_best_t, **tfidf_metrics},
    {"model": "Embeddings", "threshold": emb_best_t, **emb_metrics},
]).sort_values("score", ascending=False)

display(results_single_split)

,model,threshold,score,f1,precision,recall,roc_auc
1,LightGBM,0.02,76,0.958333,0.958333,0.958333,0.991651
2,TF-IDF,0.40,46,0.860465,0.973684,0.770833,0.934769
0,Baseline LogReg,0.78,30,0.804878,0.970588,0.687500,0.962079
3,Embeddings,0.83,28,0.784810,1.000000,0.645833,0.939529


In [13]:
# Stacking correct sur le split aléatoire :
# - méta-features d'entraînement obtenues en OOF sur le train
# - base models réentraînés sur tout le train pour prédire la validation
# - seuil choisi sur les prédictions OOF du train uniquement

stack_train_oof = generate_oof_meta_features(
    X_train_tab,
    X_train_text,
    X_train_emb,
    y_train,
    categorical_cols,
    n_splits=N_SPLITS_STACK,
    random_state=RANDOM_STATE,
)

stack_valid = fit_base_models_and_predict_valid(
    X_train_tab,
    X_valid_tab,
    X_train_text,
    X_valid_text,
    X_train_emb,
    X_valid_emb,
    y_train,
    categorical_cols,
)

stack_model = LogisticRegression(max_iter=2000, class_weight="balanced")
stack_model.fit(stack_train_oof, y_train)

stack_prob_train = stack_model.predict_proba(stack_train_oof)[:, 1]
best_t_stack, best_score_stack, threshold_df_stack = find_best_threshold(y_train, stack_prob_train)

stack_prob_valid = stack_model.predict_proba(stack_valid)[:, 1]
stack_metrics, stack_pred = compute_metrics(y_valid, stack_prob_valid, threshold=best_t_stack)

print("Best threshold (train OOF only):", round(best_t_stack, 4))
print("Best competition score (train OOF only):", best_score_stack)
display(threshold_df_stack.sort_values("score", ascending=False).head(10))

print(classification_report(y_valid, stack_pred, zero_division=0))

results_single_split = pd.concat([
    results_single_split,
    pd.DataFrame([{
        "model": "Stacking 3 modèles",
        "threshold": best_t_stack,
        **stack_metrics
    }])
], ignore_index=True).sort_values("score", ascending=False)

display(results_single_split)

  Fold interne 1/5
  Fold interne 2/5
  Fold interne 3/5
  Fold interne 4/5
  Fold interne 5/5
Best threshold (train OOF only): 0.84
Best competition score (train OOF only): 310


,threshold,score,f1,precision,recall
83,0.84,310,0.951613,0.988827,0.917098
86,0.87,308,0.948509,0.994318,0.906736
87,0.88,308,0.948509,0.994318,0.906736
85,0.86,308,0.948509,0.994318,0.906736
84,0.85,306,0.948787,0.988764,0.911917
82,0.83,304,0.949062,0.983333,0.917098
88,0.89,304,0.945652,0.994286,0.901554
89,0.90,296,0.939891,0.994220,0.891192
72,0.73,292,0.944000,0.972527,0.917098
73,0.74,292,0.944000,0.972527,0.917098


              precision    recall  f1-score   support

           0       0.98      1.00      0.99       267
           1       1.00      0.88      0.93        48

    accuracy                           0.98       315
   macro avg       0.99      0.94      0.96       315
weighted avg       0.98      0.98      0.98       315



,model,threshold,score,f1,precision,recall,roc_auc
0,LightGBM,0.02,76,0.958333,0.958333,0.958333,0.991651
4,Stacking 3 modèles,0.84,72,0.933333,1.000000,0.875000,0.984004
1,TF-IDF,0.40,46,0.860465,0.973684,0.770833,0.934769
2,Baseline LogReg,0.78,30,0.804878,0.970588,0.687500,0.962079
3,Embeddings,0.83,28,0.784810,1.000000,0.645833,0.939529


## 8. Validation leave-one-dataset-out (protocole principal)


In [14]:
dataset_results = []
all_fold_preds = []

dataset_ids = sorted(df["dataset_id"].unique())

for valid_dataset in dataset_ids:
    print("=" * 80)
    print(f"VALIDATION SUR LE DATASET {valid_dataset}")
    print("=" * 80)

    train_mask = df["dataset_id"] != valid_dataset
    valid_mask = df["dataset_id"] == valid_dataset

    X_train_tab = X_tab_full.loc[train_mask].reset_index(drop=True)
    X_valid_tab = X_tab_full.loc[valid_mask].reset_index(drop=True)

    X_train_text = X_text_all.loc[train_mask].reset_index(drop=True)
    X_valid_text = X_text_all.loc[valid_mask].reset_index(drop=True)

    X_train_emb = X_emb_all.loc[train_mask].reset_index(drop=True)
    X_valid_emb = X_emb_all.loc[valid_mask].reset_index(drop=True)

    y_train = df.loc[train_mask, TARGET_COL].astype(int).to_numpy()
    y_valid = df.loc[valid_mask, TARGET_COL].astype(int).to_numpy()

    # 1) méta-features train en OOF
    stack_train_oof = generate_oof_meta_features(
        X_train_tab,
        X_train_text,
        X_train_emb,
        y_train,
        categorical_cols,
        n_splits=N_SPLITS_STACK,
        random_state=RANDOM_STATE,
    )

    # 2) prédictions sur le dataset laissé de côté
    stack_valid = fit_base_models_and_predict_valid(
        X_train_tab,
        X_valid_tab,
        X_train_text,
        X_valid_text,
        X_train_emb,
        X_valid_emb,
        y_train,
        categorical_cols,
    )

    # 3) apprentissage du stacker et choix du seuil sur train OOF seulement
    stack_model = LogisticRegression(max_iter=2000, class_weight="balanced")
    stack_model.fit(stack_train_oof, y_train)

    train_stack_prob = stack_model.predict_proba(stack_train_oof)[:, 1]
    best_t, best_score_train, _ = find_best_threshold(y_train, train_stack_prob)

    valid_stack_prob = stack_model.predict_proba(stack_valid)[:, 1]
    metrics, y_pred = compute_metrics(y_valid, valid_stack_prob, threshold=best_t)

    print(f"Best threshold (train OOF only): {best_t:.2f}")
    print(f"Competition score: {metrics['score']:.6f}")
    print(f"F1       : {metrics['f1']:.6f}")
    print(f"Precision: {metrics['precision']:.6f}")
    print(f"Recall   : {metrics['recall']:.6f}")
    print(f"ROC-AUC  : {metrics['roc_auc']:.6f}")
    print()
    print(classification_report(y_valid, y_pred, zero_division=0))

    dataset_results.append({
        "valid_dataset": valid_dataset,
        "n_valid": int(valid_mask.sum()),
        "score": metrics["score"],
        "f1": metrics["f1"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "roc_auc": metrics["roc_auc"],
        "threshold": best_t,
    })

    fold_preds = pd.DataFrame({
        "user_id": df.loc[valid_mask, "user_id"].values,
        "dataset_id": df.loc[valid_mask, "dataset_id"].values,
        "y_true": y_valid,
        "y_prob": valid_stack_prob,
        "y_pred": y_pred,
    })
    all_fold_preds.append(fold_preds)

dataset_results_df = pd.DataFrame(dataset_results).sort_values("valid_dataset").reset_index(drop=True)
all_fold_preds_df = pd.concat(all_fold_preds, ignore_index=True)

print("\n" + "=" * 80)
print("RÉSULTATS PAR DATASET")
print("=" * 80)
display(dataset_results_df)

global_score = competition_score(all_fold_preds_df["y_true"], all_fold_preds_df["y_pred"])
global_f1 = f1_score(all_fold_preds_df["y_true"], all_fold_preds_df["y_pred"], zero_division=0)
global_precision = precision_score(all_fold_preds_df["y_true"], all_fold_preds_df["y_pred"], zero_division=0)
global_recall = recall_score(all_fold_preds_df["y_true"], all_fold_preds_df["y_pred"], zero_division=0)
global_roc_auc = roc_auc_score(all_fold_preds_df["y_true"], all_fold_preds_df["y_prob"])

print("\n" + "=" * 80)
print("SCORE GLOBAL LEAVE-ONE-DATASET-OUT")
print("=" * 80)
print(f"Competition score: {global_score:.6f}")
print(f"F1 global       : {global_f1:.6f}")
print(f"Precision global: {global_precision:.6f}")
print(f"Recall global   : {global_recall:.6f}")
print(f"ROC-AUC global  : {global_roc_auc:.6f}")

VALIDATION SUR LE DATASET 1
  Fold interne 1/5
  Fold interne 2/5
  Fold interne 3/5
  Fold interne 4/5
  Fold interne 5/5
Best threshold (train OOF only): 0.53
Competition score: 108.000000
F1       : 1.000000
Precision: 1.000000
Recall   : 1.000000
ROC-AUC  : 1.000000

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       208
           1       1.00      1.00      1.00        54

    accuracy                           1.00       262
   macro avg       1.00      1.00      1.00       262
weighted avg       1.00      1.00      1.00       262

VALIDATION SUR LE DATASET 2
  Fold interne 1/5
  Fold interne 2/5
  Fold interne 3/5
  Fold interne 4/5
  Fold interne 5/5
Best threshold (train OOF only): 0.75
Competition score: 30.000000
F1       : 0.905660
Precision: 0.923077
Recall   : 0.888889
ROC-AUC  : 0.994084

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       144
           1       0.

,valid_dataset,n_valid,score,f1,precision,recall,roc_auc,threshold
0,1,262,108,1.000000,1.000000,1.000000,1.000000,0.53
1,2,171,30,0.905660,0.923077,0.888889,0.994084,0.75
2,3,257,90,0.969697,1.000000,0.941176,0.987912,0.61
3,4,172,30,0.912281,0.896552,0.928571,0.974950,0.52
4,5,426,70,0.934579,0.925926,0.943396,0.996661,0.80
5,6,283,48,0.962963,1.000000,0.928571,0.997339,0.65



SCORE GLOBAL LEAVE-ONE-DATASET-OUT
Competition score: 376.000000
F1 global       : 0.953975
Precision global: 0.962025
Recall global   : 0.946058
ROC-AUC global  : 0.991374


## 8.bis Seuil global unique pour la production

Les validations leave-one-dataset-out donnent un ensemble de probabilités hors échantillon sur tous les utilisateurs. On peut donc choisir un **seuil global unique** pour la future production, au lieu d'utiliser un seuil différent selon les folds.


In [15]:
thresholds = np.arange(0.01, 0.50, 0.005)

best_t_global = 0.10
best_score_global = -10**18
rows_global = []

for t in thresholds:
    pred = (all_fold_preds_df["y_prob"] >= t).astype(int)

    score = competition_score(all_fold_preds_df["y_true"], pred)
    f1 = f1_score(all_fold_preds_df["y_true"], pred, zero_division=0)
    p = precision_score(all_fold_preds_df["y_true"], pred, zero_division=0)
    r = recall_score(all_fold_preds_df["y_true"], pred, zero_division=0)

    rows_global.append((t, score, f1, p, r))

    if score > best_score_global:
        best_score_global = score
        best_t_global = t

threshold_global_df = pd.DataFrame(
    rows_global,
    columns=["threshold", "score", "f1", "precision", "recall"]
)

print("Best global threshold:", best_t_global)
print("Best global competition score:", best_score_global)
display(threshold_global_df.sort_values("score", ascending=False).head(10))

Best global threshold: 0.4799999999999999
Best global competition score: 334


,threshold,score,f1,precision,recall
94,0.480,334,0.942857,0.927711,0.958506
97,0.495,330,0.940695,0.927419,0.954357
96,0.490,330,0.940695,0.927419,0.954357
95,0.485,330,0.940695,0.927419,0.954357
93,0.475,322,0.939024,0.920319,0.958506
92,0.470,316,0.937120,0.916667,0.958506
91,0.465,316,0.937120,0.916667,0.958506
90,0.460,316,0.937120,0.916667,0.958506
89,0.455,310,0.935223,0.913043,0.958506
87,0.445,304,0.933333,0.909449,0.958506


## 8. Export optionnel des résultats


In [16]:

# Décommente si tu veux conserver les sorties pour le rapport ou les expériences suivantes.
# dataset_results_df.to_csv("results_leave_one_dataset_out.csv", index=False)
# all_fold_preds_df.to_csv("predictions_leave_one_dataset_out.csv", index=False)


# Partie final soummission

1. Choix final de la solution


In [17]:
# === CONSTRUCTION DES DONNEES HISTORIQUES ===

# on utilise TOUS les datasets 1..6 pour le training final
hist_posts = posts.copy()
hist_users = users.copy()

# construction features
hist_df = build_user_level_features(hist_posts, hist_users)
hist_text_df = build_user_text_dataset(hist_posts, hist_df)
hist_embed_df = build_user_embedding_dataset(hist_posts, hist_df, embed_model, batch_size=64)

print("hist_df:", hist_df.shape)
print("hist_text_df:", hist_text_df.shape)
print("hist_embed_df:", hist_embed_df.shape)

print("Alignement hist_df / hist_text_df :", (hist_df["user_id"].values == hist_text_df["user_id"].values).all())
print("Alignement hist_df / hist_embed_df:", (hist_df["user_id"].values == hist_embed_df["user_id"].values).all())

display(hist_df.head())

hist_df: (1571, 37)
hist_text_df: (1571, 3)
hist_embed_df: (1571, 386)
Alignement hist_df / hist_text_df : True
Alignement hist_df / hist_embed_df: True


,user_id,dataset_id,dataset_lang,is_bot,username_len,username_has_digit,name_len,description_len,description_is_empty,location_len,location_is_empty,tweet_count_declared,tweet_count_log,z_score,tweet_count_observed,avg_text_len,std_text_len,avg_num_tokens,avg_num_hashtags,avg_num_mentions,avg_num_urls,lexical_diversity,duplicate_ratio,exact_duplicate_ratio,hour_entropy,night_tweet_ratio,avg_delta_seconds,std_delta_seconds,burstiness_delta,text_unique_ratio,short_tweet_ratio,url_tweet_ratio,hashtag_tweet_ratio,mention_tweet_ratio,char_entropy,first_token_repetition_ratio,avg_word_len
0,dd6de229-7892-4bec-b13d-6769c3dbd8ec,1,en,0,11,0,11,116,0,27,0,39,3.688879,0.708294,39,141.230769,82.163209,22.717949,0.000000,0.000000,0.948718,0.528217,0.0,0.0,1.868525,0.102564,4545.289474,16530.085006,0.568663,1.0,0.000000,0.948718,0.000000,0.000000,4.662888,0.153846,5.829250
1,475042a3-4676-46ad-af17-0341f1dfc58a,1,en,0,8,1,10,138,0,12,0,29,3.401197,0.188918,29,60.551724,32.641824,11.344828,0.000000,0.000000,0.068966,0.574468,0.0,0.0,2.373560,0.241379,5748.535714,13466.581326,0.401665,1.0,0.103448,0.068966,0.000000,0.000000,4.626254,0.172414,4.571830
2,d03ae074-a881-4c85-9ea0-378bf39ede7a,1,en,0,7,1,16,146,0,10,0,59,4.094345,1.747045,59,151.254237,54.063238,21.711864,1.593220,0.000000,0.694915,0.380952,0.0,0.0,3.547921,0.169492,2947.965517,8283.760957,0.475065,1.0,0.033898,0.694915,0.966102,0.000000,5.033623,0.864407,6.030782
3,eed74609-b31d-4337-ba00-a338f9f65dbe,1,en,0,11,0,43,158,0,12,0,37,3.637586,0.604419,37,48.756757,10.455583,8.054054,0.189189,0.000000,0.000000,0.436242,0.0,0.0,2.619391,0.081081,4768.472222,12539.596193,0.448989,1.0,0.000000,0.000000,0.189189,0.000000,5.103231,0.432432,4.840862
4,fbc68db2-e072-49c4-af92-94454838492d,1,en,0,11,0,12,49,0,13,0,14,2.708050,-0.590146,14,93.000000,62.517426,15.785714,0.000000,0.142857,0.357143,0.719457,0.0,0.0,2.692381,0.428571,13171.384615,20389.422101,0.215073,1.0,0.000000,0.285714,0.000000,0.142857,4.865224,0.142857,5.078089


In [18]:
# === SOLUTION FINALE : PARAMÈTRES RETENUS ===

FINAL_THRESHOLD = best_t_global
FINAL_FEATURE_COLS = FEATURE_COLS.copy()
FINAL_CATEGORICAL_COLS = categorical_cols.copy()

USE_LANG_SPECIFIC_TFIDF = True

print("FINAL_THRESHOLD =", FINAL_THRESHOLD)
print("Nombre de features =", len(FINAL_FEATURE_COLS))
print("Nombre de variables catégorielles =", len(FINAL_CATEGORICAL_COLS))
print("TF-IDF spécialisé par langue =", USE_LANG_SPECIFIC_TFIDF)

FINAL_THRESHOLD = 0.4799999999999999
Nombre de features = 33
Nombre de variables catégorielles = 0
TF-IDF spécialisé par langue = True


# === ENTRAINEMENT FINAL SUR TOUT L'HISTORIQUE LABELLISE ===


In [19]:
X_tab_train_all = hist_df[FINAL_FEATURE_COLS].copy()
X_text_train_all = hist_text_df["all_text"].copy()
X_emb_train_all = hist_embed_df.drop(columns=["user_id", TARGET_COL]).copy()
y_train_all = hist_df[TARGET_COL].astype(int).values

for col in FINAL_CATEGORICAL_COLS:
    X_tab_train_all[col] = X_tab_train_all[col].astype("category")

# ----- modèle tabulaire final
final_tab_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9,
    class_weight="balanced",
    random_state=42,
    verbose=-1,
)

final_tab_model.fit(
    X_tab_train_all,
    y_train_all,
    categorical_feature=FINAL_CATEGORICAL_COLS,
)

# ----- modèle texte global
final_text_model_global = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=30000,
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
    ))
])

final_text_model_global.fit(X_text_train_all, y_train_all)

# ----- modèles texte par langue
final_text_models_by_lang = {}

if USE_LANG_SPECIFIC_TFIDF:
    for lang in sorted(hist_df["dataset_lang"].astype(str).unique()):
        lang_idx = hist_df.index[hist_df["dataset_lang"].astype(str) == lang]

        X_lang_text = hist_text_df.loc[lang_idx, "all_text"].copy()
        y_lang = hist_df.loc[lang_idx, TARGET_COL].astype(int).values

        lang_model = Pipeline([
            ("tfidf", TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=30000,
            )),
            ("clf", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
            ))
        ])

        lang_model.fit(X_lang_text, y_lang)
        final_text_models_by_lang[lang] = lang_model

# ----- modèle embeddings final
final_emb_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
)
final_emb_model.fit(X_emb_train_all, y_train_all)

print("✔ Modèles finaux entraînés")
print("Modèles texte disponibles :", list(final_text_models_by_lang.keys()) if USE_LANG_SPECIFIC_TFIDF else ["global only"])
print("Shape X_tab_train_all :", X_tab_train_all.shape)
print("Shape X_text_train_all:", X_text_train_all.shape)
print("Shape X_emb_train_all :", X_emb_train_all.shape)

✔ Modèles finaux entraînés
Modèles texte disponibles : ['en', 'fr']
Shape X_tab_train_all : (1571, 33)
Shape X_text_train_all: (1571,)
Shape X_emb_train_all : (1571, 384)


# === PIPELINE FINAL DE SOUMISSION ===


In [20]:
LAST_DIR = Path("data/compet")

TARGET_JSON_FILES = [
     LAST_DIR / "dataset.posts&users.8.json",
]

SUBMISSION_PATH = Path("amzidou.detections.FRE.txt")

prepared_targets = []

for target_file in TARGET_JSON_FILES:
    if not target_file.exists():
        print(f"Fichier absent, ignoré : {target_file}")
        continue

    target_metadata, target_posts, target_users = load_competition_json(target_file)

    target_users = target_users.copy()
    target_users["is_bot"] = 0
    target_users["dataset_id"] = target_metadata["dataset_id"]
    target_users["dataset_lang"] = target_metadata["lang"]

    target_df = build_user_level_features(target_posts, target_users)
    target_text_df = build_user_text_dataset(target_posts, target_df)
    target_embed_df = build_user_embedding_dataset(target_posts, target_df, embed_model, batch_size=64)

    X_target_tab = target_df[FINAL_FEATURE_COLS].copy()
    for col in FINAL_CATEGORICAL_COLS:
        X_target_tab[col] = pd.Categorical(
            X_target_tab[col],
            categories=X_tab_train_all[col].cat.categories
        )

    X_target_text = target_text_df["all_text"].copy()
    X_target_emb = target_embed_df.drop(columns=["user_id", TARGET_COL]).copy()

    target_lang = str(target_metadata["lang"])

    if USE_LANG_SPECIFIC_TFIDF and target_lang in final_text_models_by_lang:
        chosen_text_model = final_text_models_by_lang[target_lang]
        chosen_text_model_name = f"tfidf_{target_lang}"
    else:
        chosen_text_model = final_text_model_global
        chosen_text_model_name = "tfidf_global"

    prepared_targets.append({
        "file": target_file,
        "metadata": target_metadata,
        "target_df": target_df,
        "X_target_tab": X_target_tab,
        "X_target_text": X_target_text,
        "X_target_emb": X_target_emb,
        "text_model": chosen_text_model,
        "text_model_name": chosen_text_model_name,
    })

print("Nombre de datasets préparés :", len(prepared_targets))

Nombre de datasets préparés : 1


# === PREDICTION + EXPORT FINAL ===


In [21]:

submission_parts = []

for item in prepared_targets:
    tab_prob_new = final_tab_model.predict_proba(item["X_target_tab"])[:, 1]
    text_prob_new = item["text_model"].predict_proba(item["X_target_text"])[:, 1]
    emb_prob_new = final_emb_model.predict_proba(item["X_target_emb"])[:, 1]

    stack_input_new = pd.DataFrame({
        "tabular_prob": tab_prob_new,
        "text_prob": text_prob_new,
        "embed_prob": emb_prob_new,
    })

    final_prob_new = stack_model.predict_proba(stack_input_new)[:, 1]
    final_prob_new = np.clip(final_prob_new, 1e-6, 1 - 1e-6)
    final_pred_new = (final_prob_new >= FINAL_THRESHOLD).astype(int)

    submission_part = item["target_df"].loc[final_pred_new == 1, ["user_id"]].copy()
    submission_part["source_file"] = item["file"].name
    submission_part["lang"] = item["metadata"]["lang"]

    print(f"Dataset: {item['file'].name}")
    print(f"Langue : {item['metadata']['lang']}")
    print(f"Modèle texte utilisé : {item['text_model_name']}")
    print(f"Bots détectés : {len(submission_part)}")
    print("-" * 60)

    submission_parts.append(submission_part)

if submission_parts:
    for part in submission_parts:
        submission_debug = pd.concat(submission_parts, ignore_index=True)
        submission = submission_debug[["user_id"]].drop_duplicates().copy()
        submission.to_csv(SUBMISSION_PATH, index=False, header=False)

    print("✔ Fichier généré :", SUBMISSION_PATH.resolve())
    print("✔ Nombre total de bots détectés :", len(submission))
    display(submission_debug.head())
else:
    print("Aucun dataset cible n'a été fourni.")

Dataset: dataset.posts&users.8.json
Langue : fr
Modèle texte utilisé : tfidf_fr
Bots détectés : 25
------------------------------------------------------------
✔ Fichier généré : C:\Users\amzid\bot_competitions\Botornobot\amzidou.detections.FRE.txt
✔ Nombre total de bots détectés : 25


,user_id,source_file,lang
0,0f582725-aa0e-4d7f-8bdd-1eff40c9224a,dataset.posts&users.8.json,fr
1,cbfc9bf6-86c2-4ccd-acfd-67bd04b2f7ed,dataset.posts&users.8.json,fr
2,6ea92ae9-5a50-44ac-83e7-d194178a9874,dataset.posts&users.8.json,fr
3,a0c6a60e-06ca-4dc3-a91f-57041cffc129,dataset.posts&users.8.json,fr
4,f02683c6-3f5d-43a6-887d-b7fa0dae65a5,dataset.posts&users.8.json,fr


# === EVALUATION RAPIDE DE LA SOLUTION FINALE SUR LE DATASET connu ===


In [22]:
# true_bot_ids = load_bot_ids(LAST_DIR / "dataset.bots.31.txt")

# eval_df = prepared_targets[0]["target_df"][["user_id"]].copy()
# eval_df["y_true"] = eval_df["user_id"].astype(str).isin(true_bot_ids).astype(int)

# pred_ids = set(pd.read_csv(SUBMISSION_PATH, header=None)[0].astype(str))
# eval_df["y_pred"] = eval_df["user_id"].astype(str).isin(pred_ids).astype(int)

# score = competition_score(eval_df["y_true"], eval_df["y_pred"])
# f1 = f1_score(eval_df["y_true"], eval_df["y_pred"], zero_division=0)
# precision = precision_score(eval_df["y_true"], eval_df["y_pred"], zero_division=0)
# recall = recall_score(eval_df["y_true"], eval_df["y_pred"], zero_division=0)

                      
# tp = ((eval_df["y_true"] == 1) & (eval_df["y_pred"] == 1)).sum()
# fn = ((eval_df["y_true"] == 1) & (eval_df["y_pred"] == 0)).sum()
# fp = ((eval_df["y_true"] == 0) & (eval_df["y_pred"] == 1)).sum()

# print("TP:", tp)
# print("FN:", fn)
# print("FP:", fp)
# print("Score check:", 2*tp - 2*fn - 6*fp)

# print("Competition score:", score)
# print("F1              :", f1)
# print("Precision       :", precision)
# print("Recall          :", recall)
# print()
# print(classification_report(eval_df["y_true"], eval_df["y_pred"], zero_division=0))